# Matching Agent - LangGraph Node Debugger

모듈화된 Matching Agent를 실행하면서 **각 LangGraph 노드가 반환한 상태 업데이트를 순서대로 확인**하는 노트북입니다.

- 필수 경로: Orchestrator → Query Reformer → Retrieve → 병렬 Selector → Main Selector → Evaluator
- 선택 경로: Message Analyzer, Web Search Module
- 재시도: Evaluator가 실패하면 Query Reformer부터 제한된 횟수만 재실행
- 주의: 실행 시 OpenAI API 비용이 발생하며, Web Search 경로에서는 Tavily API 키가 필요합니다.


In [ ]:
from pathlib import Path
import sys
import uuid

from dotenv import load_dotenv
from IPython.display import JSON, Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'matching_agent').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env')

from matching_agent import build_matching_graph, get_initial_state
from matching_agent.config import DEFAULT_RECURSION_LIMIT

print('Project root:', PROJECT_ROOT)


## 1. 그래프 생성 및 구조 확인

In [ ]:
graph = build_matching_graph()
mermaid = graph.get_graph().draw_mermaid()
display(Markdown(f'```mermaid\n{mermaid}\n```'))


## 2. 입력 설정

`thread_id`는 checkpoint를 구분하는 실행 ID입니다. 셀을 다시 실행할 때마다 새 UUID를 사용합니다.

In [ ]:
INPUT_MESSAGE = '고등학생 밴드에서 일렉기타 연주자를 급하게 찾고 있어요.'
USERNAME = 'debug_user'
THREAD_ID = f'debug-{uuid.uuid4()}'

initial_state = get_initial_state(INPUT_MESSAGE, USERNAME)
config = {
    'configurable': {'thread_id': THREAD_ID},
    'recursion_limit': DEFAULT_RECURSION_LIMIT,
}

print('Thread ID:', THREAD_ID)
print('Input:', INPUT_MESSAGE)


## 3. 노드별 응답 스트리밍

LangGraph `stream_mode="updates"`를 사용해 각 노드가 실제로 변경한 필드만 표시합니다. TypeMatch, RoleMatch, PersonaMatch는 같은 super-step에서 병렬 실행되므로 출력 순서는 달라질 수 있습니다.

In [ ]:
def _message_content(message):
    return getattr(message, 'content', str(message))


def _compact_update(update):
    if not isinstance(update, dict):
        return {'value': str(update)}

    visible_keys = [
        'next_agent',
        'last_agent',
        'matching_target',
        'hypernym',
        'analyzed_message',
        'search_info',
        'reformed_queries',
        'messages',
        'selector_results',
        'matched_username',
        'matched_message',
        'certainty',
        'fail_or_not',
        'evaluation_count',
        'retry_count',
        'failure_log',
    ]
    compact = {key: update[key] for key in visible_keys if key in update}

    if update.get('candidate_hits'):
        compact['candidate_hits'] = [
            {
                'username': hit['username'],
                'message': hit['message'],
                'sources': hit['sources'],
            }
            for hit in update['candidate_hits']
        ]

    if update.get('history'):
        compact['node_answer'] = _message_content(update['history'][-1])
    if update.get('trace'):
        compact['trace'] = update['trace']
    return compact


def show_node_update(node_name, update):
    display(Markdown(f'### `{node_name}`'))
    display(JSON(_compact_update(update), expanded=True))


In [ ]:
streamed_updates = []

for part in graph.stream(
    initial_state,
    config=config,
    stream_mode='updates',
    version='v2',
):
    if part['type'] != 'updates':
        continue
    for node_name, update in part['data'].items():
        streamed_updates.append({'node': node_name, 'update': update})
        show_node_update(node_name, update)


## 4. 최종 결과 요약

In [ ]:
snapshot = graph.get_state(config)
final_state = snapshot.values

final_summary = {
    'input_message': final_state.get('input_message'),
    'matching_target': final_state.get('matching_target'),
    'hypernym': final_state.get('hypernym'),
    'reformed_queries': final_state.get('reformed_queries'),
    'candidate_count': len(final_state.get('candidate_hits', [])),
    'matched_username': final_state.get('matched_username'),
    'matched_message': final_state.get('matched_message'),
    'certainty': final_state.get('certainty'),
    'result': final_state.get('fail_or_not'),
    'evaluation_count': final_state.get('evaluation_count'),
    'retry_count': final_state.get('retry_count'),
    'failure_log': final_state.get('failure_log'),
}
display(JSON(final_summary, expanded=True))


## 5. Checkpoint 이력 확인

각 super-step 뒤 저장된 checkpoint를 최신 순서로 확인합니다. 재시도가 발생했다면 Query Reformer 이후 구간이 반복된 것을 볼 수 있습니다.

In [ ]:
checkpoint_history = list(graph.get_state_history(config))

for index, checkpoint in enumerate(checkpoint_history):
    display(Markdown(f'### Checkpoint {index}'))
    display(JSON({
        'next_nodes': list(checkpoint.next),
        'last_agent': checkpoint.values.get('last_agent'),
        'evaluation_count': checkpoint.values.get('evaluation_count'),
        'retry_count': checkpoint.values.get('retry_count'),
        'matched_username': checkpoint.values.get('matched_username'),
    }, expanded=True))


## 참고

- 다른 입력은 `INPUT_MESSAGE`와 `USERNAME`만 변경한 뒤 2번 셀부터 다시 실행합니다.
- 같은 `graph` 객체에서 여러 실험을 할 때는 반드시 새로운 `THREAD_ID`를 사용합니다.
- 노트북은 관찰·디버깅용이며 실제 구현의 기준은 `matching_agent/` 모듈입니다.
- API 키는 출력 셀이나 Git 저장소에 기록하지 마세요.